# Tiny English Chatbot Language Model From Scratch (Google Colab GPU)

This beginner-friendly notebook trains a very small GPT-2 style English chatbot language model from scratch.

What it does:

1. Mounts Google Drive.
2. Creates or loads a JSONL English conversation dataset with 5,000 examples.
3. Trains a small BPE tokenizer and saves it to Google Drive.
4. Builds a GPT-2 style transformer with fewer than 10 million parameters.
5. Trains for 5 epochs using AdamW.
6. Prints training and validation loss after each epoch.
7. Generates a sample reply to a test prompt.
8. Saves the trained model and tokenizer to Google Drive.

> **Before running:** In Colab, choose **Runtime → Change runtime type → GPU**.

In [ ]:
# If you are running in Google Colab, this cell installs the libraries we need.
# - transformers: model classes and training utilities
# - tokenizers: fast BPE tokenizer training
# - datasets is not required here because we keep the dataset simple for beginners
!pip -q install transformers tokenizers accelerate

In [ ]:
# Mount Google Drive so that your dataset, tokenizer, and trained model are saved permanently.
# Colab will ask you to authorize access to your Drive.
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# Import the Python libraries used throughout the notebook.
import json
import math
import os
import random
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast, get_linear_schedule_with_warmup

# Set a random seed so runs are more repeatable.
random.seed(42)
torch.manual_seed(42)

# Use GPU if Colab provides one; otherwise fall back to CPU.
# Training will be much faster with a GPU runtime.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Create folders in Google Drive for this project.
# Everything important will be saved under PROJECT_DIR.
PROJECT_DIR = Path('/content/drive/MyDrive/tiny_english_chatbot')
DATA_DIR = PROJECT_DIR / 'data'
TOKENIZER_DIR = PROJECT_DIR / 'tokenizer_bpe'
MODEL_DIR = PROJECT_DIR / 'tiny_gpt2_chatbot_model'

for folder in [DATA_DIR, TOKENIZER_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / 'english_conversations_5000.jsonl'
print('Project folder:', PROJECT_DIR)

In [ ]:
# Create or load a small English conversation dataset in JSONL format.
# JSONL means "JSON Lines": each line is one separate JSON object.
# Each example has a "prompt" and a "reply" field.
#
# If the file already exists in Google Drive, we load it.
# If it does not exist, we generate a simple synthetic English-only dataset.
# This keeps the notebook self-contained and ready to run.

def build_synthetic_conversations(min_examples=5000):
    greetings = [
        ('hello', 'Hello! How can I help you today?'),
        ('hi there', 'Hi! I am happy to chat with you.'),
        ('good morning', 'Good morning! I hope your day is starting well.'),
        ('good evening', 'Good evening! What would you like to talk about?'),
        ('how are you', 'I am doing well and ready to help.'),
    ]

    topics = {
        'weather': ['sunny weather', 'rainy days', 'cold mornings', 'warm afternoons', 'cloudy skies'],
        'food': ['apples', 'sandwiches', 'soup', 'pasta', 'salad'],
        'school': ['homework', 'reading', 'math practice', 'science class', 'history notes'],
        'hobbies': ['drawing', 'music', 'walking', 'gardening', 'reading books'],
        'technology': ['computers', 'phones', 'robots', 'websites', 'keyboards'],
        'health': ['sleep', 'water', 'exercise', 'stretching', 'rest'],
        'travel': ['trains', 'maps', 'hotels', 'airports', 'city parks'],
        'pets': ['dogs', 'cats', 'birds', 'fish', 'rabbits'],
        'work': ['meetings', 'emails', 'projects', 'planning', 'teamwork'],
        'kindness': ['helping others', 'sharing', 'listening', 'patience', 'thank you notes'],
    }

    prompt_templates = [
        'Can you tell me about {item}?',
        'What do you think about {item}?',
        'I want to learn about {item}.',
        'Do you have advice about {item}?',
        'Please say something simple about {item}.',
        'Why do people like {item}?',
        'How can I get better at {item}?',
        'Give me a friendly thought about {item}.',
    ]

    reply_templates = [
        '{item_cap} can be interesting when you take it one step at a time.',
        'A simple way to think about {item} is to stay curious and keep practicing.',
        'Many people enjoy {item} because it can make daily life feel better.',
        'For {item}, start small, be patient, and notice what works for you.',
        'I think {item} is a good topic for a calm and friendly conversation.',
        'When learning about {item}, asking clear questions is a great beginning.',
    ]

    examples = []
    for prompt, reply in greetings:
        examples.append({'prompt': prompt, 'reply': reply})

    for topic, items in topics.items():
        for item in items:
            for p_template in prompt_templates:
                for r_template in reply_templates:
                    examples.append({
                        'prompt': p_template.format(item=item),
                        'reply': r_template.format(item=item, item_cap=item.capitalize()),
                    })

    # Add some short multi-turn style examples.
    feelings = ['happy', 'sad', 'tired', 'excited', 'nervous', 'confused', 'hopeful', 'bored']
    for feeling in feelings:
        examples.extend([
            {'prompt': f'I feel {feeling} today.', 'reply': f'Thank you for telling me. It is okay to feel {feeling}, and I am here to listen.'},
            {'prompt': f'What should I do when I feel {feeling}?', 'reply': 'Take a slow breath, name one small next step, and be kind to yourself.'},
        ])

    random.shuffle(examples)
    # Repeat the template-generated examples with small variations until we reach the requested size.
    # This keeps the notebook self-contained while providing 5,000 training examples.
    expanded = []
    round_number = 1
    while len(expanded) < max(min_examples, 1000):
        for ex in examples:
            if round_number == 1:
                expanded.append(ex)
            else:
                expanded.append({
                    'prompt': ex['prompt'],
                    'reply': ex['reply'] + f' This is example variation {round_number}.',
                })
            if len(expanded) >= max(min_examples, 1000):
                break
        round_number += 1

    random.shuffle(expanded)
    return expanded[:max(min_examples, 1000)]

if DATASET_PATH.exists():
    print('Loading existing dataset:', DATASET_PATH)
else:
    print('Creating a new synthetic English conversation dataset...')
    examples = build_synthetic_conversations(min_examples=5000)
    with DATASET_PATH.open('w', encoding='utf-8') as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + '\n')

# Load the dataset and show a few examples.
conversations = []
with DATASET_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        conversations.append(json.loads(line))

print('Number of examples:', len(conversations))
print('First 3 examples:')
for ex in conversations[:3]:
    print(ex)

In [ ]:
# Convert prompt/reply pairs into text strings for language-model training.
# The model will learn the pattern:
# User: ...
# Bot: ...
#
# The special token <|endoftext|> marks where one example ends.
train_texts = [
    f"User: {ex['prompt']}\nBot: {ex['reply']}<|endoftext|>"
    for ex in conversations
]

TOKENIZER_TRAIN_FILE = DATA_DIR / 'tokenizer_training_text.txt'
with TOKENIZER_TRAIN_FILE.open('w', encoding='utf-8') as f:
    for text in train_texts:
        f.write(text + '\n')

print('Prepared tokenizer training text:', TOKENIZER_TRAIN_FILE)

In [ ]:
# Train a simple Byte Pair Encoding (BPE) tokenizer from scratch.
# A tokenizer turns text into numbers that the model can understand.
# We keep the vocabulary small because this is a tiny beginner model.
VOCAB_SIZE = 2000

bpe_tokenizer = ByteLevelBPETokenizer()
bpe_tokenizer.train(
    files=[str(TOKENIZER_TRAIN_FILE)],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=['<|endoftext|>', '<|pad|>'],
)
bpe_tokenizer.save_model(str(TOKENIZER_DIR))

# Load the saved tokenizer using the Transformers GPT2TokenizerFast wrapper.
tokenizer = GPT2TokenizerFast.from_pretrained(
    str(TOKENIZER_DIR),
    bos_token='<|endoftext|>',
    eos_token='<|endoftext|>',
    unk_token='<|endoftext|>',
    pad_token='<|pad|>',
)
tokenizer.save_pretrained(str(TOKENIZER_DIR))

print('Tokenizer vocabulary size:', len(tokenizer))
print('Tokenizer saved to:', TOKENIZER_DIR)

In [ ]:
# Build a small GPT-2 style transformer language model from scratch.
# This model is intentionally tiny and has fewer than 10 million parameters.
# Bigger models usually produce better text, but they need more data, memory, and training time.
MAX_LENGTH = 64

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=MAX_LENGTH,
    n_ctx=MAX_LENGTH,
    n_embd=192,
    n_layer=6,
    n_head=6,
    bos_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model = GPT2LMHeadModel(config).to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {num_params:,}')
assert num_params < 10_000_000, 'Model must be under 10M parameters.'

In [ ]:
# Create a PyTorch Dataset.
# For language modeling, the input IDs and labels are the same sequence.
# The model learns to predict the next token at each position.
class ChatDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.examples = []
        for text in texts:
            encoded = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )
            input_ids = encoded['input_ids'].squeeze(0)
            attention_mask = encoded['attention_mask'].squeeze(0)

            # Labels are copied from input_ids.
            # Padding tokens use label -100 so PyTorch ignores them in the loss.
            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            self.examples.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

# Split the data into 90% training and 10% validation.
# Validation data is not used for learning; it helps us choose the best model.
random.shuffle(train_texts)
split_index = int(len(train_texts) * 0.9)
train_split_texts = train_texts[:split_index]
val_split_texts = train_texts[split_index:]

train_dataset = ChatDataset(train_split_texts, tokenizer, MAX_LENGTH)
val_dataset = ChatDataset(val_split_texts, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print('Training examples:', len(train_dataset))
print('Validation examples:', len(val_dataset))
print('Training batches per epoch:', len(train_loader))
print('Validation batches per epoch:', len(val_loader))

In [ ]:
# Train the model for 5 epochs with the AdamW optimizer.
# AdamW is a common optimizer for transformer language models.
# After each epoch, we measure validation loss and save the best model so far.
EPOCHS = 5
LEARNING_RATE = 5e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, total_steps // 10),
    num_training_steps=total_steps,
)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # Validation checks how well the model predicts examples it did not train on.
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_val_loss += outputs.loss.item()

    average_val_loss = total_val_loss / len(val_loader)
    print(f'Epoch {epoch}/{EPOCHS} - training loss: {average_train_loss:.4f} - validation loss: {average_val_loss:.4f}')

    # Save the best model based on validation loss, not simply the final epoch.
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        model.save_pretrained(str(MODEL_DIR))
        tokenizer.save_pretrained(str(TOKENIZER_DIR))
        print(f'New best model saved with validation loss: {best_val_loss:.4f}')

In [ ]:
# Generate a sample reply after training.
# Because this is a tiny model trained on a small synthetic dataset for only 5 epochs,
# the reply may be simple or imperfect. That is normal for a first model.
model.eval()

def generate_reply(prompt, max_new_tokens=40):
    formatted_prompt = f'User: {prompt}\nBot:'
    inputs = tokenizer(formatted_prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return generated_text

sample_prompt = 'Can you tell me about reading books?'
print(generate_reply(sample_prompt))

In [ ]:
# The best model was already saved during training whenever validation loss improved.
# Save the tokenizer again here and print the final locations.
# You can reload them later with:
#   GPT2LMHeadModel.from_pretrained(MODEL_DIR)
#   GPT2TokenizerFast.from_pretrained(TOKENIZER_DIR)
tokenizer.save_pretrained(str(TOKENIZER_DIR))

print('Best model saved to:', MODEL_DIR)
print('Tokenizer saved to:', TOKENIZER_DIR)
print('Dataset saved to:', DATASET_PATH)
print(f'Best validation loss: {best_val_loss:.4f}')

## Next steps

This notebook is intentionally small so it can run in Colab and be understandable for beginners. To improve the chatbot later, you can:

- Use a larger and more natural conversation dataset.
- Train for more epochs.
- Increase the model size carefully.
- Add validation data to measure overfitting.
- Experiment with generation settings such as `temperature`, `top_k`, and `top_p`.